# Crowd counting — corrected baselines and MCNN hyperparameter search

**Open this notebook in Google Colab with a GPU runtime. Upload the accompanying `crowd-hyperparameters-source.zip` when prompted.** This source snapshot contains the new, possibly uncommitted code; a Git clone/pull alone is not sufficient. The original `TreniramCeoDan.ipynb` attachment is preserved outside this repository; it is not required by this notebook.

## Operation order (explicit opt-in; Run all does not train)
1. Edit the configuration below, then run setup/upload/install/GPU verification.
2. Enable `RUN_DATA_PREP` and `RUN_PRECOMPUTE`, run their cells, then turn them off.
3. Inspect the dry-run trial list/budget. Enable the baseline flags and run their cells if desired.
4. Enable `RUN_SEARCH` and run the search cell. It resumes incomplete trials and verifies/skips completed ones, including the baseline.
5. Inspect **validation-only** ranking. Optionally enable `RUN_CONFIRMATION` and run confirmation for the fixed baseline and winner on seeds 123 and 2026.
6. Only when the configuration and seed plan are final, set `EVALUATE_WITH_CONFIRMATION` appropriately and enable `RUN_FINAL_EVALUATION`. This explicitly opens the test split; never use its metrics to choose hyperparameters.
7. Export the generated real JSON/CSV/plots and update the project report later. Running this notebook is not itself evidence that the experiments finished.

**Budget:** 3 learning rates × 2 momenta × 2 weight decays = 12 MCNN trials, 50 epochs each, Part A, batch 4, seed 42 by default. Baseline is included, not a thirteenth run. Optional confirmation adds at most four runs (baseline and winner × two additional seeds); identical baseline/winner shares checkpoints. CSRNet has a separate fixed baseline, not a search. Wall time is hardware-dependent; no runtime promise is made.

**Original outputs inspected:** baseline MCNN validation MAE 2629.85 at epoch 50 was still decreasing. The separate lr=1e-5 pilot reached validation MAE 464.74 and test MAE/RMSE 315.2272/429.1577, overwriting the baseline filename in the original workflow. CSRNet output ended at epoch 48; it did not demonstrate completion. These are historical notebook observations, not new results. Independent review replaced 3e-7 with 1e-5 using the historical **validation** evidence, preserving the baseline and 12-trial budget. The default rates are now [1e-6, 3e-6, 1e-5]; the historically stronger setting participates in validation-only selection, without promising improvement. Use a new experiment ID for this revised grid, never alter it after inspecting test scores. Fifty epochs is a fixed-budget comparison, not proof of convergence. The official test split has already been inspected in the pilot; it is excluded from selection in this new search, not claimed to be globally unseen.

## Završeni rezultati u repozitorijumu
MCNN: svih 12 pokušaja i 4 potvrde završeno je po 50 epoha; izabrano lr=1e-5, momentum=.95, weight_decay=1e-4 samo po validaciji. Part A test seed 42: MAE/RMSE 316.3083/431.9880. Zaseban lokalni CSRNet seed 42: 233.4080/376.9634, najbolja epoha 31. Tri MCNN seed-a prikazana su posebno u [finalnom izveštaju](../reports/report.md); nema Part B ili dodatnih CSRNet rezultata.
Originalni laki dokazi su u punom repozitorijumu pod `reports/evidence/`, a `scripts/plot_training_curves.py` reprodukuje grafikone bez treninga. Training-only ZIP namerno ne sadrži te dokaze, final-report skriptu ni njene testove; relativni link ka izveštaju važi u punom repozitorijumu. Ova napomena nije novi izvršeni izlaz ćelija; originalni priloženi notebook ostaje neizmenjen. Za novi eksperiment koristiti novi izlazni direktorijum i proveriti plan pre pristupa testu.


In [ ]:
from pathlib import Path
import json

PART = "A"  # "B" works too; choose a different EXPERIMENT_ID for a new study
EPOCHS = 50
BATCH_SIZE = 4
SEARCH_SEED = 42
CONFIRMATION_SEEDS = [123, 2026]
GRID = {"lr": [1e-6, 3e-6, 1e-5], "momentum": [0.90, 0.95], "weight_decay": [0.0, 1e-4]}
BASELINE = {"lr": 1e-6, "momentum": 0.95, "weight_decay": 0.0}
CSRNET_SEEDS = [42]  # set [42, 123, 2026] before starting for three-seed model comparison
EXPERIMENT_ID = f"mcnn-grid-part{PART}-v2"
DRIVE_ROOT = Path("/content/drive/MyDrive/crowd-hyperparameters")
BUNDLE_ON_DRIVE = DRIVE_ROOT / "crowd-hyperparameters-source.zip"
EXPECTED_BUNDLE_SHA256 = None  # exporter pins this in the delivered notebook
DATASET_ON_DRIVE = Path("/content/drive/MyDrive/ShanghaiTech")

RUN_DATA_PREP = False
RUN_PRECOMPUTE = False
RUN_MCNN_BASELINE = False
RUN_CSRNET_BASELINE = False
RUN_SEARCH = False
RUN_CONFIRMATION = False
EVALUATE_WITH_CONFIRMATION = False  # set True only after all confirmation runs finish
EVALUATE_CSRNET = False  # opt in only when every CSRNET_SEEDS run is complete
RUN_FINAL_EVALUATION = False
RUN_EXPORT = False


## 1. Mount Drive and restore/upload the exact source snapshot
No recursive deletion and no nested relative `cd`. Each bundle gets an absolute, content-addressed `/content` directory. Only upload a bundle you trust: it contains executable Python code. The manifest detects accidental corruption, not a malicious bundle author. Keep the same bundle and experiment ID after disconnects; changed code/config must use a new experiment/output directory. Drive-mounted rename/fsync reduces partial local writes, but does not guarantee cloud synchronization: check files in Drive before deleting a runtime.

In [ ]:
from google.colab import drive, files
import hashlib
import io
import os
import shutil
import stat
import subprocess
import sys
import zipfile

drive.mount("/content/drive")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
if BUNDLE_ON_DRIVE.is_file():
    bundle_bytes = BUNDLE_ON_DRIVE.read_bytes()
else:
    uploaded = files.upload()
    candidates = [data for name, data in uploaded.items() if name.endswith(".zip")]
    if len(candidates) != 1:
        raise RuntimeError("Upload exactly one crowd-hyperparameters-source.zip")
    bundle_bytes = candidates[0]
bundle_sha = hashlib.sha256(bundle_bytes).hexdigest()
if EXPECTED_BUNDLE_SHA256 and bundle_sha != EXPECTED_BUNDLE_SHA256:
    raise RuntimeError("Wrong source bundle. Set BUNDLE_ON_DRIVE to a new path and upload the zip paired with this notebook.")
PROJECT = Path("/content") / ("crowd-hyperparameters-source-" + bundle_sha[:16])
with zipfile.ZipFile(io.BytesIO(bundle_bytes)) as archive:
    names = archive.namelist()
    if len(names) != len(set(names)):
        raise RuntimeError("Duplicate archive entries")
    manifest = json.loads(archive.read("BUNDLE_MANIFEST.json"))
    if set(names) != set(manifest["files"]) | {"BUNDLE_MANIFEST.json"}:
        raise RuntimeError("Unexpected archive files")
    for info in archive.infolist():
        name = info.filename
        if Path(name).is_absolute() or ".." in Path(name).parts or stat.S_ISLNK(info.external_attr >> 16):
            raise RuntimeError("Unsafe archive entry: " + name)
        data = archive.read(name)
        if name != "BUNDLE_MANIFEST.json" and hashlib.sha256(data).hexdigest() != manifest["files"][name]:
            raise RuntimeError("Bundle checksum mismatch: " + name)
        dest = PROJECT / name
        if dest.exists() and dest.read_bytes() != data:
            raise RuntimeError("Existing source differs; do not overwrite: " + str(dest))
        dest.parent.mkdir(parents=True, exist_ok=True)
        if not dest.exists():
            dest.write_bytes(data)
if not BUNDLE_ON_DRIVE.exists():
    BUNDLE_ON_DRIVE.write_bytes(bundle_bytes)
os.chdir(PROJECT)
OUTPUT = DRIVE_ROOT / EXPERIMENT_ID
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Project:", PROJECT, "\nOutput:", OUTPUT, "\nBundle SHA256:", bundle_sha)
print("Snapshot metadata:", {k: v for k, v in manifest.items() if k != "files"})


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "sync", "--locked"], cwd=PROJECT, check=True)

def run(arguments, log_name="notebook.log"):
    # shell=False and exit-code checking: errors cannot disappear behind a tee pipe.
    command = ["uv", "run", "--locked", "python", "-u", *map(str, arguments)]
    log = OUTPUT / "logs" / log_name
    log.parent.mkdir(parents=True, exist_ok=True)
    with log.open("a") as handle:
        handle.write("\nCOMMAND " + json.dumps(command) + "\n")
        with subprocess.Popen(command, cwd=PROJECT, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                handle.write(line)
                handle.flush()
            status = process.wait()
    if status:
        raise RuntimeError(f"Command failed ({status}); see {log}")

run(["-c", "import torch; from src.config import CACHE_VERSION; "
     "assert CACHE_VERSION == 2, 'Expected corrected density-cache v2'; "
     "assert torch.cuda.is_available(), 'Select a Colab GPU runtime before real training'; "
     "print('torch', torch.__version__, 'CUDA', torch.version.cuda, 'GPU', torch.cuda.get_device_name(0))"], "environment.log")
run(["-m", "pytest", "tests", "-q"], "tests.log")


## 2. Dataset restore/download and cache preparation
The complete dataset is backed up to Drive. A partial local dataset is refused rather than silently trained. The train/validation split is the fixed final 10% of lexicographically sorted official training filenames (A: 270/30; B: 360/40); it is identical across seeds, not stratified/randomized. No test images are used for training or selection. Cache generation is not training; this cell precomputes **train and validation only**. Test maps are generated during explicit final evaluation. Keep the original dataset immutable during a search.

In [ ]:
DATASET = PROJECT / "data/raw/ShanghaiTech"
CACHE = PROJECT / "data/processed/density_maps"

def dataset_complete(root):
    for part, counts in {"A": (300, 182), "B": (400, 316)}.items():
        for split, count in zip(["train_data", "test_data"], counts):
            folder = root / f"part_{part}" / split
            images = sorted((folder / "images").glob("*.jpg"))
            gts = sorted((folder / "ground-truth").glob("*.mat"))
            if len(images) != count or len(gts) != count:
                return False
            if {"GT_" + p.stem for p in images} != {p.stem for p in gts}:
                return False
    return True

if RUN_DATA_PREP:
    if not dataset_complete(DATASET):
        if DATASET.exists():
            raise RuntimeError("Partial local dataset: inspect/move it manually, then rerun. Nothing deleted.")
        if dataset_complete(DATASET_ON_DRIVE):
            shutil.copytree(DATASET_ON_DRIVE, DATASET)
        else:
            run(["scripts/download_data.py"], "download.log")
    assert dataset_complete(DATASET), "Download/restore incomplete"
    if not DATASET_ON_DRIVE.exists():
        shutil.copytree(DATASET, DATASET_ON_DRIVE)
    assert dataset_complete(DATASET_ON_DRIVE), "Drive backup incomplete; repair it before training"
    print("Dataset verified locally and on Drive")
else:
    print("Dataset step disabled; enable RUN_DATA_PREP for first setup")

if RUN_PRECOMPUTE:
    assert dataset_complete(DATASET), "Prepare dataset first"
    run(["scripts/precompute_density_maps.py", "--parts", PART, "--models", "mcnn", "csrnet",
         "--splits", "train", "val", "--root", DATASET, "--cache-dir", CACHE], "precompute.log")


## 3. Freeze configuration and inspect budget (no training)
Trial IDs are hashes of model/part/seed/budget/optimizer settings. Search protocol also binds the source fingerprint; completed runs bind data hashes and preprocessing. Changed settings do not silently overwrite old checkpoints. The optimizer is SGD, pixel-wise mean MSE, no scheduler/early stopping/augmentation; train `drop_last=True` retains the existing protocol. Training-loss means now divide by samples actually processed.

Outputs go directly to Drive: config, per-epoch history, summary, best weights, `last.pth` (optimizer/RNG/history/best snapshot), and hashed completion manifest. Re-running resumes at the next completed epoch, not mid-batch. A failed/non-finite trial prevents winner selection until resolved. Run only one process per output directory. Exact bitwise reproducibility across different GPU models/PyTorch versions is not promised.

In [ ]:
SEARCH_CONFIG = {"model": "mcnn", "part": PART, "seed": SEARCH_SEED, "epochs": EPOCHS,
                 "batch_size": BATCH_SIZE, "num_workers": 0, "grid": GRID,
                 "baseline": BASELINE, "confirmation_seeds": CONFIRMATION_SEEDS}
CONFIG_PATH = OUTPUT / "search_config.json"
if CONFIG_PATH.exists() and json.loads(CONFIG_PATH.read_text()) != SEARCH_CONFIG:
    raise RuntimeError("Configuration changed: choose a new EXPERIMENT_ID")
CONFIG_PATH.write_text(json.dumps(SEARCH_CONFIG, indent=2))
SEARCH_OUT = OUTPUT / "search"
SEARCH_ARGS = ["-m", "src.search", "--config", CONFIG_PATH, "--out-dir", SEARCH_OUT,
               "--root", DATASET, "--cache-dir", CACHE, "--device", "cuda"]
run([*SEARCH_ARGS, "--stage", "dry-run"], "budget.log")


## 4. Corrected baselines (optional separate execution)
The MCNN baseline is one of the 12 search trials; running it here avoids waiting for grid order. The full search later skips it only after verifying hashes/config. CSRNet remains lr=1e-5, momentum=.95, weight_decay=0 with pretrained VGG16 frontend. Different optimizers/seeds never share a filename directory. The historical MCNN lr=1e-5 setting is now included in the grid; no duplicate reference run is needed.

In [ ]:
if RUN_MCNN_BASELINE:
    run([*SEARCH_ARGS, "--stage", "baseline"], "mcnn-baseline.log")

def train_baseline(model, seed, lr, folder):
    assert dataset_complete(DATASET), "Prepare data first"
    run(["-m", "src.train", "--model", model, "--part", PART, "--seed", seed,
         "--epochs", EPOCHS, "--batch-size", BATCH_SIZE, "--lr", lr, "--momentum", .95,
         "--weight-decay", 0, "--root", DATASET, "--cache-dir", CACHE,
         "--out-dir", folder, "--resume", "--device", "cuda"], f"{model}-seed{seed}-{folder.name}.log")

if RUN_CSRNET_BASELINE:
    for seed in CSRNET_SEEDS:
        train_baseline("csrnet", seed, 1e-5, OUTPUT / "csrnet" / f"seed{seed}")


## 5. Run all MCNN grid trials and freeze validation-only selection
Selection uses **best validation MAE only** over the equal epoch budget. Ties are resolved by trial ID ascending, not by test metrics or validation RMSE. All trials must have valid completion manifests. Review `search_results.csv`, `selection.json`, and `validation_curves.png`. Do not interpret synthetic/local smoke checks as ShanghaiTech results.

In [ ]:
if RUN_SEARCH:
    assert dataset_complete(DATASET), "Prepare data first"
    run([*SEARCH_ARGS, "--stage", "search"], "search.log")
if (SEARCH_OUT / "selection.json").exists():
    selection = json.loads((SEARCH_OUT / "selection.json").read_text())
    print("Frozen winner:", json.dumps(selection["winner"], indent=2))
    from IPython.display import display, Image
    display(Image(filename=str(SEARCH_OUT / "validation_curves.png")))
else:
    print("No selection yet; enable RUN_SEARCH to execute the grid")


## 6. Optional additional seeds for the fixed baseline and winner
These runs estimate training-seed variability; they do not reselect hyperparameters. Keep the same fixed train/validation split. If baseline equals winner, the runner reuses identical runs. Three-seed aggregation includes seed 42 from the original search and seeds 123/2026 from this stage.

In [ ]:
if RUN_CONFIRMATION:
    run([*SEARCH_ARGS, "--stage", "confirm"], "confirmation.log")


## 7. Explicit final test evaluation and report
Choose the seed plan **before** enabling this cell. The runner freezes `evaluation_plan.json` (selected configurations, seed list, checkpoint hashes) before reading any test images; changing from single-seed to confirmation evaluation in the same output directory is refused. Sample standard deviation uses n−1; for one seed it is null, not zero. Test is never used to select the winner. Set EVALUATE_CSRNET=True to include all declared CSRNET_SEEDS. Every requested run must finish; missing seeds are never silently skipped. The separate seed/checkpoint plan is frozen before any test access, including the MCNN stage. CSRNet results never alter the winner. Only trusted project-generated checkpoints should be loaded (PyTorch pickle).

In [ ]:
if RUN_FINAL_EVALUATION:
    extra = ["--with-confirmation"] if EVALUATE_WITH_CONFIRMATION else []
    # Other predeclared baselines are independent of search selection.
    import hashlib
    if len(set(CSRNET_SEEDS)) != len(CSRNET_SEEDS) or not CSRNET_SEEDS:
        raise RuntimeError("Declare unique nonempty CSRNET_SEEDS before evaluation")
    other_dirs = [("csrnet", OUTPUT / "csrnet" / f"seed{seed}") for seed in CSRNET_SEEDS] if EVALUATE_CSRNET else []
    plan = {"runs": []}
    for (model, folder), seed in zip(other_dirs, CSRNET_SEEDS):
        if not (folder / "complete.json").exists():
            raise RuntimeError("Finish all requested CSRNet seeds before any test evaluation")
        run(["-c", "from src.runs import completed; import sys; assert completed(sys.argv[1])", folder], "verify-baselines.log")
        summary = json.loads((folder / "summary.json").read_text())
        expected = {"model": model, "part": PART, "seed": seed, "epochs": EPOCHS,
                    "batch_size": BATCH_SIZE, "lr": 1e-5, "momentum": .95, "weight_decay": 0, "smoke": False}
        if any(summary["config"].get(k) != v for k, v in expected.items()):
            raise RuntimeError("CSRNet configuration differs from declared baseline")
        checkpoint_hash = hashlib.sha256((folder / summary["checkpoint"]).read_bytes()).hexdigest()
        plan["runs"].append({"model": model, "seed": seed, "checkpoint_hash": checkpoint_hash})
    plan_path = OUTPUT / "other_evaluation_plan.json"
    if plan_path.exists() and json.loads(plan_path.read_text()) != plan:
        raise RuntimeError("Separate baseline evaluation plan changed; use a new experiment")
    # Atomic helper runs inside uv's environment, not the Colab kernel.
    run(["-c", "from src.search import lock_json; from pathlib import Path; import json,sys; "
         "lock_json(Path(sys.argv[1]), json.loads(sys.argv[2]))", plan_path, json.dumps(plan)], "freeze-baselines.log")
    run([*SEARCH_ARGS, "--stage", "evaluate", *extra], "final-evaluation.log")
    other_results = []
    for model, folder in other_dirs:
        run(["-c", "from src.runs import completed; import sys; assert completed(sys.argv[1])", folder], "verify-baselines.log")
        summary = json.loads((folder / "summary.json").read_text())
        run(["-m", "src.evaluate", "--model", model, "--part", PART,
             "--ckpt", folder / summary["checkpoint"], "--root", DATASET,
             "--cache-dir", CACHE, "--out", folder / "test_metrics.json", "--device", "cuda"], "other-evaluation.log")
        metrics = json.loads((folder / "test_metrics.json").read_text())
        other_results.append({"label": "csrnet",
                              "seed": metrics["seed"], "mae": metrics["mae"], "rmse": metrics["rmse"]})
    import statistics
    groups = {}
    for label in sorted({r["label"] for r in other_results}):
        values = [r for r in other_results if r["label"] == label]
        groups[label] = {"n": len(values), "seeds": [r["seed"] for r in values]}
        for metric in ("mae", "rmse"):
            samples = [r[metric] for r in values]
            groups[label][metric + "_mean"] = statistics.mean(samples)
            groups[label][metric + "_sample_std"] = statistics.stdev(samples) if len(samples) > 1 else None
    (OUTPUT / "other_baseline_metrics.json").write_text(json.dumps({"runs": other_results, "aggregate": groups}, indent=2, allow_nan=False))
    print((SEARCH_OUT / "aggregate.json").read_text())
else:
    print("Test evaluation disabled")


## 8. Export real results (not checkpoints/datasets)
All raw artifacts already live in Drive. Download this compact report archive after the real run, review it, then copy real summary CSV/JSON/plots into the repository's reports folder for the final analysis. No commit or push is performed. Preserve the source zip, protocol/config, and full checkpoints on Drive for reproduction. Colab execution itself was not performed by the notebook author; run the cells in your actual GPU runtime.

In [ ]:
if RUN_EXPORT:
    if not (SEARCH_OUT / "selection.json").exists():
        raise RuntimeError("Finish search first")
    run([*SEARCH_ARGS, "--stage", "report"], "report.log")
    export = OUTPUT / "crowd-hyperparameters-results.zip"
    with zipfile.ZipFile(export, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(OUTPUT.rglob("*")):
            if path.is_file() and path.suffix in {".json", ".csv", ".png"}:
                archive.write(path, path.relative_to(OUTPUT))
    files.download(str(export))
    print("Exported", export, "— add only reviewed real results to the project report")
